In [ ]:
from ase.io import read

atoms_1000K = read("data/Van_Vleck/1000K_out_selected.xyz", ":")
atoms_400K = read("data/Van_Vleck/400K_out_selected.xyz", ":")

In [ ]:
import numpy as np

q_vectors_1000K = np.array([[[atom_1000K.arrays["Q2_automated"][site.index], atom_1000K.arrays["Q3_1_automated"][site.index]] for site in atom_1000K if site.symbol == "Mn"] for atom_1000K in atoms_1000K])
positions_1000K = np.array([[site.position for site in atom_1000K if site.symbol == "Mn"] for atom_1000K in atoms_1000K])
q_vectors_400K = np.array([[[atom_400K.arrays["Q2_automated"][site.index], atom_400K.arrays["Q3_1_automated"][site.index]] for site in atom_400K if site.symbol == "Mn"] for atom_400K in atoms_400K])
positions_400K = np.array([[site.position for site in atom_400K if site.symbol == "Mn"] for atom_400K in atoms_400K])

In [ ]:
def autocorr_gr(vectors):
    vectors = np.asarray(vectors)
    T, N, d = vectors.shape
    max_dt = T
    G = np.zeros(max_dt)
    G_std = np.zeros(max_dt)

    for dt in range(0, max_dt):
        vec_t0 = vectors[0]
        vec_t1 = vectors[dt]
        dot_products = np.zeros(N)
        for i, (t0_pair, t1_pair) in enumerate(zip(vec_t0, vec_t1)):
            if np.any(np.isnan(t0_pair)) or np.any(np.isnan(t1_pair)):
                continue
            dot_product = np.dot(t0_pair, t1_pair)
            dot_products[i] = dot_product
        G[dt] = np.mean(dot_products)
        G_std[dt] = np.std(dot_products)
    return G, G_std

In [ ]:
G_auto_1000K, G_auto_1000K_std = autocorr_gr(q_vectors_1000K)
G_auto_400K, G_auto_400K_std = autocorr_gr(q_vectors_400K)

In [ ]:
plt.plot(G_auto_400K, marker='o', linestyle='-', color='black', label='400K')
plt.plot(G_auto_1000K, marker='o', linestyle='-', color='red', label='1000K')
plt.fill_between(range(len(G_auto_400K)), G_auto_400K - G_auto_400K_std, G_auto_400K + G_auto_400K_std, color='black', alpha=0.2)
plt.fill_between(range(len(G_auto_1000K)), G_auto_1000K - G_auto_1000K_std, G_auto_1000K + G_auto_1000K_std, color='red', alpha=0.2)

plt.hlines(0, 0, len(G_auto_400K), colors='black', linestyles='dashed', linewidth=1, alpha=0.5)

plt.xlabel("Time (fs)")
plt.ylabel("G(t)")
plt.legend(frameon=False)
plt.tight_layout()
plt.xlim(0, 100)
plt.savefig("autocorr_gr_plot.svg", dpi=600)
plt.show()